# Sztuczna Inteligencja – Laboratorium nr 3
## Rozpoznawanie odręcznych cyfr – MNIST
**ANS – Sztuczna Inteligencja | dr inż. Jacek Paluszak**

---
Zbiór danych MNIST zawiera 70 000 obrazów (28x28 pikseli) z odręcznymi cyframi.
Celem jest zbudowanie sieci neuronowej z 2 ukrytymi warstwami do klasyfikacji 10 cyfr (0–9).

## Blok 1 – Import bibliotek

In [ ]:
# Biblioteka numpy ułatwia pracę z macierzami i wielowymiarowymi tabelami.
import numpy as np
# Biblioteka Tensorflow posłuży nam do zbudowania modelu
import tensorflow as tf
# Biblioteka opencv-python - biblioteka typu open source do przetwarzania obrazu i uczenia maszynowego.
import cv2
# Wizualizacja danych.
import matplotlib.pyplot as plt
# TensorFlow zawiera dostawcę danych dla MNIST, z którego będziemy korzystać.
import tensorflow_datasets as tfds

print('TensorFlow version:', tf.__version__)
print('OpenCV version:', cv2.__version__)
print('NumPy version:', np.__version__)

## Blok 2 – Załadowanie i wstępne przetworzenie danych

In [ ]:
# tfds.load w rzeczywistości ładuje zestaw danych (lub pobiera, a następnie ładuje, jeśli używasz go po raz pierwszy)
# w naszym przypadku interesuje nas MNIST; nazwa zbioru danych jest jedynym obowiązkowym argumentem
# istnieją inne argumenty, które możemy określić i które mogą nam się przydać
# with_info = True dostarczy nam również krótką zawierającą informacje o wersji, funkcjach, liczbie próbek
# wykorzystamy te informacje nieco poniżej i zapiszemy je w mnist_info
# as_supervised = True załaduje zestaw danych w strukturze 2-krotkowej (wejście, cel)
# alternatywnie, as_supervised = False, zwróci słownik
# oczywiście wolimy, aby nasze dane wejściowe i cele były oddzielone
mnist_dataset, mnist_info = tfds.load(name='mnist', with_info=True, as_supervised=True)


# Po załadowaniu zestawu danych możemy łatwo wyodrębnić zestaw danych szkoleniowy
# i testowych z utworzonymi referencjami
mnist_train, mnist_test = mnist_dataset['train'], mnist_dataset['test']

# Domyślnie TF ma zestawy danych treningowych i testowych, ale nie ma zestawów walidacyjnych
# dlatego musimy go samodzielnie podzielić
# zaczynamy od zdefiniowania liczby próbek walidacyjnych jako % próbek trenujących
# tutaj również używamy mnist_info (nie musimy liczyć obserwacji)
num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples

# sparsujmy to na liczbę całkowitą, ponieważ liczba zmiennoprzecinkowa może po drodze spowodować błąd
num_validation_samples = tf.cast(num_validation_samples, tf.int64)

# zapiszmy też liczbę próbek testowych w dedykowanej zmiennej (zamiast używać mnist_info)
num_test_samples = mnist_info.splits['test'].num_examples

# jeszcze raz parsujemy na liczbę całkowitą (zamiast domyślnej liczby zmiennoprzecinkowej)
num_test_samples = tf.cast(num_test_samples, tf.int64)

print('Liczba próbek walidacyjnych:', num_validation_samples.numpy())
print('Liczba próbek testowych:', num_test_samples.numpy())

## Blok 3 – Normalizacja danych

In [ ]:
# należy znormalizować nasze dane, aby wynik był bardziej stabilny numerycznie
# dane wejściowe muszą być w zakresie liczbowym od 0 do 1
# zdefiniujmy funkcję o nazwie: scale, która pobierze obraz MNIST i jego etykietę

def scale(image, label):
    # potrzebujemy wartości zmiennoprzecinkowej float
    image = tf.cast(image, tf.float32)
    # ponieważ możliwe wartości dla wejść to od 0 do 255 (256 różnych odcieni szarości)
    # jeśli podzielimy każdy element przez 255, otrzymamy pożądany wynik:
    # wszystkie elementy będą znajdować się w przedziale od 0 do 1
    image /= 255.
    return image, label

# metoda .map() pozwala nam zastosować niestandardową transformację do danego zbioru danych
# już zdecydowaliśmy, że otrzymamy dane walidacyjne z mnist_train, więc
scaled_train_and_validation_data = mnist_train.map(scale)

# na koniec normalizujemy i grupujemy dane testowe
# aby miały taką samą wielkość jak dane trenujące i walidacyjne
# nie ma potrzeby ich mieszania, ponieważ nie będziemy trenować na danych testowych
# będzie to pojedyncza partia równa wielkości danych testowych
test_data = mnist_test.map(scale)

print('Dane znormalizowane.')

## Blok 4 – Mieszanie i podział danych

In [ ]:
# ten parametr BUFFER_SIZE jest tutaj w przypadkach, gdy mamy do czynienia z ogromnymi zbiorami danych
# wtedy nie możemy przetworzyć całego zestawu danych za jednym razem,
# ponieważ nie możemy zmieścić go w pamięci
# więc zamiast tego TF przechowuje w pamięci tylko BUFFER_SIZE próbki na raz i tasuje je
# if BUFFER_SIZE = 1 => żadne tasowanie nie nastąpi
# if BUFFER_SIZE> = liczba próbek => tasowanie jest jednolite
# BUFFER_SIZE pomiędzy - optymalizacja obliczeniowa w celu przybliżenia jednolitego tasowania
BUFFER_SIZE = 1000

# jest łatwo dostępna metoda shuffle i musimy tylko określić rozmiar bufora
shuffled_train_and_validation_data = scaled_train_and_validation_data.shuffle(BUFFER_SIZE)

# po przeskalowaniu i przetasowaniu danych możemy przystąpić do faktycznego wyodrębniania
# danych trenujących i walidacyjnych
# nasze dane walidacyjne byłyby równe 10% zbioru uczącego, który już obliczyliśmy
# używamy metody .take(), aby pobrać tyle próbek
# na koniec tworzymy partię o wielkości równej całkowitej liczbie próbek walidacyjnych
validation_data = shuffled_train_and_validation_data.take(num_validation_samples)

# podobnie train_data to wszystko inne, więc pomijamy tyle próbek
# ile jest w zbiorze danych walidacyjnych
train_data = shuffled_train_and_validation_data.skip(num_validation_samples)

print('Dane podzielone na treningowe i walidacyjne.')

## Blok 5 – Tworzenie partii (batch)

In [ ]:
# zmienna określająca wielkość partii podawanej podczas trenowania
BATCH_SIZE = 100

# tworzymy partie danych treningowych
# jest to bardzo pomocne podczas treningu, ponieważ możemy iterować po różnych partiach
train_data = train_data.batch(BATCH_SIZE)

validation_data = validation_data.batch(num_validation_samples)

test_data = test_data.batch(num_test_samples)

# pobieramy i iterujemy partię walidacyjną (jest to jedyna partia)
# ponieważ as_supervised = True, mamy strukturę składającą się z dwóch krotek
validation_inputs, validation_targets = next(iter(validation_data))
print(validation_inputs.shape, validation_targets.shape)

## Blok 6 – Budowanie modelu

In [ ]:
input_size = 784  # 28 x 28 pikseli
output_size = 10  # 10 różnych cyfr

# Użyjemy tego samego rozmiaru ukrytej warstwy dla obu ukrytych warstw. Nie jest to konieczne.
hidden_layer_size = 50

#MODEL
model = tf.keras.Sequential([

    # Pierwsza warstwa to warstwa wejściowa.
    # Każda próbka ma wymiary 28x28x1 pikseli, dlatego jest to tensor rozmiaru 3.
    # Ponieważ to nie jest jeszcze CNN, nie wiemy, jak wprowadzić takie dane wejściowe
    # do naszej sieci, więc musimy spłaszczyć obrazy.
    # istnieje wygodna warstwa "Flatten", która po prostu pobiera nasz tensor 28x28x1
    # i porządkuje go w (None,) lub (28x28x1,) = (784,)
    # to pozwala nam faktycznie stworzyć sieć neuronową typu feed forward
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),  # input layer, warstwa wejściowa

    # tf.keras.layers.Dense jest w zasadzie implementacją modelu liniowego:
    # y = xw + b  czyli   output = activation(dot(input, weight) + bias)
    # wymaga kilku argumentów, ale najważniejsze dla nas to hidden_layer_size i funkcja aktywacji
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),  # warstwa ukryta
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),  # warstwa ukryta

    # ostatnia warstwa nie jest inna, po prostu upewniamy się,
    # że aktywujemy ją za pomocą softmax, która daje nam rozkład prawdopodobieństwa
    # i ma rozmiar output_size
    tf.keras.layers.Dense(output_size, activation='softmax')  # output layer
])

# określamy na końcu funkcję optymalizacji, którego chcielibyśmy użyć,
# funkcja straty, # oraz metryki, które chcemy uzyskać w każdej epocy uczenia
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.summary()

## Blok 7 – Trenowanie modelu

In [ ]:
# ustawiamy ilość epok
NUM_EPOCHS = 30

# ustaw mechanizm wczesnego zatrzymywania, który ochroni nasz model przed przetrenowaniem
# ustawmy patience=2, aby być nieco tolerancyjnym na losowe wzrosty strat walidacji
early_stopping = tf.keras.callbacks.EarlyStopping(patience=2)

# dopasowujemy model, określając dane treningowe, łączną liczbę epok
# oraz dane walidacyjne, które właśnie stworzyliśmy, w formacie: (INPUTS, TARGETS),

model.fit(train_data,  # dane wejściowe trenujące
          epochs=NUM_EPOCHS,  # maksymalna ilość epok gdyby wczesne zatrzymanie nie zadziałało
          callbacks=[early_stopping],  # mechanizm "early stopping" - zapobieganie przetrenowaniu
          validation_data=(validation_inputs, validation_targets),  # dane walidacyjne
          verbose=1  # sposób pokazania treningu modelu
          )

## Blok 8 – Testowanie modelu

In [ ]:
test_loss, test_accuracy = model.evaluate(test_data)
# możemy zastosować estetyczne formatowanie
print('Test loss: {0:.2f}. Test accuracy: {1:.2f}%'.format(test_loss, test_accuracy*100.))

---
## ZADANIE – Predykcja własnego obrazu cyfry

Wykorzystując bibliotekę `opencv-python` oraz metodę `model.predict()`, ładujemy własny obrazek cyfry pisanej odręcznie i sprawdzamy skuteczność modelu.

### Jak przygotować obrazek:
- Obrazek może mieć dowolny rozmiar – zostanie przeskalowany do 28x28 px
- Powinien być **biała cyfra na czarnym tle** (tak jak w MNIST)
- Jeśli masz **czarną cyfrę na białym tle**, zostanie automatycznie odwrócony
- Możesz wgrać swój plik do Google Colab przez panel boczny (ikona folderu)

> **Uwaga:** Pamiętaj o PONOWNYM URUCHOMIENIU wszystkich bloków kodu po zmianach!

In [ ]:
def predict_digit_from_image(image_path, model, show_image=True):
    """
    Wczytuje obrazek cyfry, przetwarza go do formatu MNIST
    i zwraca predykcję modelu.

    Parametry:
        image_path (str): Ścieżka do pliku obrazka (np. 'cyfra.png')
        model: Wytrenowany model Keras
        show_image (bool): Czy wyświetlić podgląd obrazka

    Zwraca:
        predicted_digit (int): Przewidywana cyfra (0-9)
        confidence (float): Pewność predykcji w procentach
    """

    # --- Krok 1: Wczytanie obrazka przez OpenCV ---
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f'Nie znaleziono pliku: {image_path}')

    # --- Krok 2: Konwersja do skali szarości ---
    # MNIST używa obrazów jednokanałowych (grayscale)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # --- Krok 3: Przeskalowanie do 28x28 pikseli (format MNIST) ---
    img_resized = cv2.resize(img_gray, (28, 28), interpolation=cv2.INTER_AREA)

    # --- Krok 4: Sprawdzenie czy cyfra jest biała na czarnym tle ---
    # W MNIST cyfry są BIAŁE na CZARNYM tle.
    # Jeśli nasz obrazek ma odwrócone kolory (czarna cyfra na białym tle),
    # należy go odwrócić za pomocą bitwise_not.
    mean_val = np.mean(img_resized)
    if mean_val > 127:  # jasne tło = odwróć kolory
        img_resized = cv2.bitwise_not(img_resized)
        print('Wykryto jasne tło – kolory zostały automatycznie odwrócone.')

    # --- Krok 5: Normalizacja (tak samo jak podczas trenowania) ---
    # Wartości pikseli z [0, 255] → [0.0, 1.0]
    img_normalized = img_resized.astype('float32') / 255.0

    # --- Krok 6: Dopasowanie kształtu do wejścia modelu ---
    # Model oczekuje kształtu: (batch_size, 28, 28, 1)
    # np.expand_dims dodaje wymiary: (28,28) → (1, 28, 28, 1)
    img_input = np.expand_dims(img_normalized, axis=(0, -1))  # shape: (1, 28, 28, 1)

    # --- Krok 7: Predykcja ---
    # model.predict() zwraca tablicę prawdopodobieństw dla każdej klasy (0-9)
    predictions = model.predict(img_input, verbose=0)  # shape: (1, 10)

    # Wybieramy klasę z najwyższym prawdopodobieństwem
    predicted_digit = int(np.argmax(predictions[0]))
    confidence = float(np.max(predictions[0])) * 100

    # --- Krok 8: Wyświetlenie obrazka i wyniku ---
    if show_image:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))

        # Oryginalny obrazek
        axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[0].set_title('Oryginalny obrazek')
        axes[0].axis('off')

        # Przetworzony obraz (28x28, grayscale) – jak widzi go model
        axes[1].imshow(img_resized, cmap='gray')
        axes[1].set_title(f'Obraz po przetworzeniu (28x28)\nPredykcja: {predicted_digit}  |  Pewność: {confidence:.1f}%')
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()

        # Wykres prawdopodobieństw dla wszystkich klas
        fig2, ax = plt.subplots(figsize=(8, 3))
        bars = ax.bar(range(10), predictions[0] * 100, color=['#2196F3']*10)
        bars[predicted_digit].set_color('#F44336')  # wyróżnienie przewidywanej cyfry
        ax.set_xlabel('Cyfra')
        ax.set_ylabel('Prawdopodobieństwo [%]')
        ax.set_title('Rozkład prawdopodobieństwa predykcji')
        ax.set_xticks(range(10))
        ax.set_ylim(0, 105)
        for i, v in enumerate(predictions[0] * 100):
            if v > 0.5:
                ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=8)
        plt.tight_layout()
        plt.show()

    print(f'\n=== WYNIK PREDYKCJI ===')
    print(f'Przewidywana cyfra: {predicted_digit}')
    print(f'Pewność: {confidence:.2f}%')

    return predicted_digit, confidence

In [ ]:
# =====================================================
# TUTAJ WPISZ ŚCIEŻKĘ DO SWOJEGO OBRAZKA
# =====================================================
# Jeśli wgrałeś plik do Colaba (np. przez panel boczny),
# podaj jego nazwę poniżej. Przykłady:
#   image_path = 'cyfra_3.png'
#   image_path = '/content/moja_cyfra.jpg'

image_path = 'cyfra.png'  # <- ZMIEŃ NA SWOJĄ ŚCIEŻKĘ

# Uruchomienie predykcji
predicted, conf = predict_digit_from_image(image_path, model, show_image=True)

In [ ]:
# =====================================================
# OPCJONALNIE: Testowanie wielu obrazków naraz
# =====================================================
# Jeśli chcesz przetestować kilka obrazków jednocześnie,
# wpisz ich ścieżki poniżej:

obrazki = [
    ('cyfra_3.png', 3),   # (ścieżka, oczekiwana_cyfra)
    ('cyfra_5.png', 5),
    ('cyfra_9.png', 9),
]

print('=== TEST WIELU OBRAZKÓW ===\n')
correct = 0
total = len(obrazki)

for path, expected in obrazki:
    try:
        pred, conf = predict_digit_from_image(path, model, show_image=True)
        is_correct = pred == expected
        correct += int(is_correct)
        status = '✓ POPRAWNA' if is_correct else f'✗ BŁĘDNA (oczekiwano: {expected})'
        print(f'Plik: {path} → Predykcja: {pred} ({conf:.1f}%)  {status}\n')
    except FileNotFoundError as e:
        print(f'[POMINIĘTO] {e}\n')

if total > 0:
    print(f'\nSkuteczność na własnych obrazkach: {correct}/{total} = {correct/total*100:.0f}%')